# TrialLens experiments on Kaggle

Trains two LoRA variants at the same time, one per T4 GPU, so design choices can be compared against the r16 baseline run.

| Run | Change from r16 | Question it answers |
|---|---|---|
| `os40` | oversample rare labels to at least 40 examples per epoch | does label imbalance explain the zero F1 on `not included` and `excluded`? |
| `r8` | LoRA rank 8, alpha 16 | does a smaller adapter generalise as well with half the parameters? |

Same setup as `kaggle_train.ipynb`: **GPU T4 x2**, Internet on, `GITHUB_TOKEN` secret attached. Use Save Version, Save & Run All so it runs in the background.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if not os.path.exists("/kaggle/working/TrialLens"):
    !git clone -q https://{token}@github.com/AbdullahRasheed452/TrialLens.git /kaggle/working/TrialLens
%cd /kaggle/working/TrialLens
!git pull -q
!git log --oneline -3

def check():
    assert _exit_code == 0, f"command failed with exit code {_exit_code}"

In [ ]:
!pip uninstall -y -q torchao
!pip install -q transformers==5.20.2 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0
check()
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/make_split.py
check()
!python -m pytest -q
check()

In [ ]:
# Each run gets its own GPU through CUDA_VISIBLE_DEVICES and writes its own log file
runs = {
    "os40": "--oversample-floor 40",
    "r8": "--lora-r 8 --lora-alpha 16",
}
os.makedirs("logs", exist_ok=True)
procs = {}
for gpu, (name, extra) in enumerate(runs.items()):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = f"python scripts/train_lora.py --run-name {name} --epochs 4 {extra} > logs/{name}.log 2>&1"
    procs[name] = subprocess.Popen(cmd, shell=True, env=env)
    print("started", name, "on GPU", gpu)

for name, p in procs.items():
    p.wait()
    print()
    print(f"===== {name} finished with exit code {p.returncode} =====")
    lines = open(f"logs/{name}.log", encoding="utf-8", errors="replace").read().splitlines()
    for line in lines:
        if "val_macro_f1" in line or "best epoch" in line or "Error" in line:
            print(line)
assert all(p.returncode == 0 for p in procs.values()), "a run failed, see logs/"


In [ ]:
!zip -qr /kaggle/working/triallens_experiments.zip logs checkpoints/os40/lora-best checkpoints/os40/train_log.json checkpoints/r8/lora-best checkpoints/r8/train_log.json
check()
!ls -lh /kaggle/working/triallens_experiments.zip